In [ ]:

# trinetx_cosmos_analysis.py
"""
Comprehensive literature analysis for TriNetX and Epic Cosmos studies.
Fetches articles from PubMed, enriches with OpenAlex metrics, extracts metadata,
and generates visualizations.
"""

import requests
import time
from lxml import etree
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

# ── Debug / fast-iteration mode ──────────────────────────────────────────
TEST_MODE  = False   # Set to False for a full production run
TEST_LIMIT = 100     # Number of articles to process in test mode

BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
EMAIL = "<>"  # Replace with your email for NCBI polite pool

QUERIES = {

    # TriNetX: add [ot] (author-supplied keywords) to [tiab]
    "trinetx": 'TriNetX[tiab] OR TriNetX[ot]',

    # Epic Cosmos: phrase match as anchor; add fallback for non-phrase references
    # The fallback uses "Epic" + "Cosmos" + healthcare context to filter noise
    "cosmos": (
        '"Epic Cosmos"[tiab] OR '
        '"Epic Cosmos"[ot] OR '
        '(Cosmos[tiab] AND Epic[tiab] AND '
        '(EHR[tiab] OR "electronic health record"[tiab] OR '
        '"real-world data"[tiab] OR "real world data"[tiab] OR '
        '"real-world evidence"[tiab] OR "patient data"[tiab] OR '
        '"claims data"[tiab] OR "clinical data"[tiab]))'
    ),
}

CORRECTION_TYPES = {
    "RetractionIn": "RETRACTED",
    "CommentIn": "COMMENT_IN",
    "LetterIn": "LETTER_IN",
    "ErratumIn": "ERRATUM",
    "ExpressionOfConcernIn": "EXPRESSION_OF_CONCERN",
}

EMPTY_METRICS = {
    "oa_display_name": None,
    "citedness_2yr": None,
    "h_index": None,
    "works_count": None,
}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PUBMED FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def txt(el, path):
    """Safe text extraction from lxml element."""
    node = el.find(path)
    return node.text.strip() if node is not None and node.text else None


def esearch(query, retmax=5000):
    """Search PubMed and return web environment for batch fetching."""
    r = requests.get(f"{BASE_URL}/esearch.fcgi", params={
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "usehistory": "y",
        "retmode": "json",
        "email": EMAIL,
    })
    r.raise_for_status()
    res = r.json()["esearchresult"]
    return res["webenv"], res["querykey"], int(res["count"])


def efetch_batch(webenv, query_key, start, retmax=200):
    """Fetch batch of PubMed records as XML."""
    r = requests.get(f"{BASE_URL}/efetch.fcgi", params={
        "db": "pubmed",
        "WebEnv": webenv,
        "query_key": query_key,
        "retstart": start,
        "retmax": retmax,
        "rettype": "xml",
        "retmode": "xml",
        "email": EMAIL,
    })
    r.raise_for_status()
    return etree.fromstring(r.content)


def extract_countries(article):
    """
    Extract author countries from affiliation strings.
    Returns list of country names found in affiliations.
    """
    countries = []

    # Get all affiliation strings
    for aff in article.findall(".//Affiliation"):
        if aff.text:
            aff_text = aff.text
            # Common country patterns - expand as needed
            country_patterns = {
                r'\bUSA\b|\bUnited States\b|\bU\.S\.A\b': 'USA',
                r'\bUK\b|\bUnited Kingdom\b|\bEngland\b|\bScotland\b|\bWales\b': 'UK',
                r'\bChina\b': 'China',
                r'\bCanada\b': 'Canada',
                r'\bGermany\b': 'Germany',
                r'\bFrance\b': 'France',
                r'\bItaly\b': 'Italy',
                r'\bSpain\b': 'Spain',
                r'\bAustralia\b': 'Australia',
                r'\bJapan\b': 'Japan',
                r'\bIndia\b': 'India',
                r'\bBrazil\b': 'Brazil',
                r'\bNetherlands\b': 'Netherlands',
                r'\bSweden\b': 'Sweden',
                r'\bSwitzerland\b': 'Switzerland',
                r'\bSouth Korea\b|\bKorea\b': 'South Korea',
            }

            for pattern, country in country_patterns.items():
                if re.search(pattern, aff_text, re.IGNORECASE):
                    countries.append(country)
                    break  # Take first match per affiliation

    return countries


def extract_keywords(article):
    """Extract MeSH terms and keywords from article."""
    keywords = []

    # MeSH terms
    for mesh in article.findall(".//MeshHeading/DescriptorName"):
        if mesh.text:
            keywords.append(mesh.text)

    # Author keywords
    for kw in article.findall(".//Keyword"):
        if kw.text:
            keywords.append(kw.text)

    return keywords


def parse_article(article):
    """
    Extract comprehensive metadata from PubmedArticle XML element.
    Includes: core metadata, corrections status, countries, and keywords.
    """
    pmid = txt(article, ".//PMID")
    title = txt(article, ".//ArticleTitle")
    year = txt(article, ".//PubDate/Year") or txt(article, ".//PubDate/MedlineDate")

    # Extract just year from MedlineDate if needed (e.g., "2020 Jan-Feb")
    if year and not year.isdigit():
        year_match = re.search(r'\d{4}', year)
        year = year_match.group() if year_match else None

    journal = txt(article, ".//Journal/Title")
    issn = (
        article.findtext(".//ISSN[@IssnType='Electronic']")
        or article.findtext(".//ISSN[@IssnType='Print']")
    )
    doi = next(
        (a.text for a in article.findall(".//ArticleId") if a.get("IdType") == "doi"),
        None,
    )
    abstract = " ".join(
        a.text for a in article.findall(".//AbstractText") if a.text
    ) or None

    # Parse CommentsCorrections for retractions, errata, comments
    flags = set()
    for c in article.findall(".//CommentsCorrections"):
        label = CORRECTION_TYPES.get(c.get("RefType", ""))
        if label:
            flags.add(label)
    pubmed_status = ", ".join(sorted(flags)) if flags else "Clean"

    # Extract countries and keywords
    countries = extract_countries(article)
    keywords = extract_keywords(article)

    return {
        "pmid": pmid,
        "title": title,
        "year": year,
        "journal": journal,
        "issn": issn,
        "doi": doi,
        "abstract": abstract,
        "pubmed_status": pubmed_status,
        "countries": "; ".join(countries) if countries else None,
        "primary_country": countries[0] if countries else None,
        "keywords": "; ".join(keywords) if keywords else None,
    }

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# OPENALEX FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def get_openalex_metrics_batch(issns):
    """Fetch journal metrics for up to 50 ISSNs in a single request."""
    if not issns:
        return {}

    filter_str = "issn:" + "|".join(issns)

    try:
        r = requests.get(
            "https://api.openalex.org/sources",
            params={"filter": filter_str, "per_page": 50, "mailto": EMAIL},
            timeout=15,
        )
    except Exception as e:
        print(f"\n  ⚠ Batch journal metrics request failed: {e}")
        return {}

    if r.status_code == 429:
        return None                         # Signal caller to retry

    if r.status_code != 200:
        print(f"\n  ⚠ OpenAlex sources returned {r.status_code}")
        return {}

    metrics = {}
    for venue in r.json().get("results", []):
        # Collect all ISSNs associated with this venue
        venue_issns = set(venue.get("issn") or [])
        if venue.get("issn_l"):
            venue_issns.add(venue["issn_l"])

        stats = venue.get("summary_stats", {})
        m = {
            "oa_display_name": venue.get("display_name"),
            "citedness_2yr":   stats.get("2yr_mean_citedness"),
            "h_index":         stats.get("h_index"),
            "works_count":     venue.get("works_count"),
        }
        # Map result back to whichever input ISSNs matched this venue
        for issn in issns:
            if issn in venue_issns:
                metrics[issn] = m

    return metrics


def get_openalex_citations_batch(pmids):
    """Fetch citation counts for up to 50 PMIDs in a single request."""
    if not pmids:
        return {}

    pmid_urls = [f"https://pubmed.ncbi.nlm.nih.gov/{pmid}" for pmid in pmids]
    filter_str = "pmid:" + "|".join(pmid_urls)

    try:
        r = requests.get(
            "https://api.openalex.org/works",
            params={"filter": filter_str, "per_page": 50, "mailto": EMAIL},
            timeout=15,
        )
    except Exception as e:
        print(f"\n  ⚠ Batch citation request failed: {e}")
        return {}

    if r.status_code == 429:
        return None                         # Signal caller to retry

    if r.status_code != 200:
        print(f"\n  ⚠ OpenAlex works returned {r.status_code}")
        return {}

    citations = {}
    for work in r.json().get("results", []):
        pmid_url = (work.get("ids") or {}).get("pmid")
        if pmid_url:
            pmid = pmid_url.rstrip("/").split("/")[-1]   # extract numeric ID
            citations[pmid] = work.get("cited_by_count", 0)

    return citations


def enrich_with_openalex(df):
    """Add journal metrics and citation counts using batched OpenAlex requests."""
    print("\n" + "=" * 70)
    print("ENRICHING WITH OPENALEX")
    print("=" * 70)

    BATCH_SIZE = 50

    # ── Journal metrics ───────────────────────────────────────────────────
    unique_issns  = df["issn"].dropna().unique().tolist()
    issn_batches  = [unique_issns[i:i+BATCH_SIZE] for i in range(0, len(unique_issns), BATCH_SIZE)]

    print(f"\nJournal metrics: {len(unique_issns)} ISSNs → {len(issn_batches)} batches")

    journal_metrics = {}
    for i, batch in enumerate(issn_batches, 1):
        print(f"  Batch {i}/{len(issn_batches)}...", end="\r")
        for attempt in range(3):
            result = get_openalex_metrics_batch(batch)
            if result is None:
                wait = 5 * (attempt + 1)
                print(f"\n  ⚠ Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                journal_metrics.update(result)
                break
        time.sleep(0.15)

    metrics_df = df["issn"].map(
        lambda x: journal_metrics.get(x, EMPTY_METRICS.copy())
    ).apply(pd.Series)
    df = pd.concat([df, metrics_df], axis=1)

    # ── Citation counts ───────────────────────────────────────────────────
    pmids        = df["pmid"].dropna().tolist()
    pmid_batches = [pmids[i:i+BATCH_SIZE] for i in range(0, len(pmids), BATCH_SIZE)]

    print(f"\n\nCitation counts: {len(pmids)} articles → {len(pmid_batches)} batches")

    citation_counts = {}
    for i, batch in enumerate(pmid_batches, 1):
        print(f"  Batch {i}/{len(pmid_batches)}...", end="\r")
        for attempt in range(3):
            result = get_openalex_citations_batch(batch)
            if result is None:
                wait = 5 * (attempt + 1)
                print(f"\n  ⚠ Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                citation_counts.update(result)
                break
        time.sleep(0.15)

    df["cited_by_count"] = df["pmid"].map(citation_counts)

    # ── Coverage report ───────────────────────────────────────────────────
    print(f"\n✓ Journal metrics:  {df['citedness_2yr'].notna().sum()}/{len(df)} articles")
    print(f"✓ Citation counts:  {df['cited_by_count'].notna().sum()}/{len(df)} articles")
    print("✓ Enrichment complete")

    return df

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA COLLECTION
# ═══════════════════════════════════════════════════════════════════════════

def fetch_all_articles():
    """Fetch all TriNetX and Epic Cosmos articles from PubMed."""
    print("=" * 70)
    print("FETCHING ARTICLES FROM PUBMED")
    print("=" * 70)

    records = []
    for source, query in QUERIES.items():
        webenv, qkey, count = esearch(query)
        print(f"\n{source.upper()}: {count} articles found")

        for start in range(0, count, 200):
            print(f"  Fetching articles {start+1}-{min(start+200, count)}...", end="\r")
            root = efetch_batch(webenv, qkey, start)
            for art in root.findall(".//PubmedArticle"):
                row = parse_article(art)
                row["source_query"] = source
                records.append(row)
            time.sleep(0.11)  # Respect NCBI rate limit
        print()  # New line after progress

    # Deduplicate on PMID, combining source labels
    raw = pd.DataFrame(records)
    source_combined = (
        raw.groupby("pmid")["source_query"]
        .apply(lambda x: "+".join(sorted(set(x))))
        .reset_index()
    )
    df = (
        raw.drop_duplicates("pmid")
        .drop(columns="source_query")
        .merge(source_combined, on="pmid")
    )

    print(f"\n✓ Total unique articles: {len(df)}")
    print(f"✓ Source breakdown:")
    print(df["source_query"].value_counts().to_string())

    if TEST_MODE:
        df = df.head(TEST_LIMIT)
        print(f"\n⚠  TEST MODE: truncated to {TEST_LIMIT} articles for fast iteration")
        print(f"   Set TEST_MODE = False in config for a full run\n")

    return df

In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# TOPIC CATEGORIZATION
# ═══════════════════════════════════════════════════════════════════════════

def categorize_topics(df):
    """
    Categorize articles by topic using keyword matching.
    """
    print("\n" + "=" * 70)
    print("CATEGORIZING TOPICS")
    print("=" * 70)

    topic_categories = {
        "COVID-19/Pandemic": [
            "covid", "sars-cov-2", "coronavirus", "pandemic", "omicron", "delta variant"
        ],
        "Cardiovascular": [
            "heart", "cardiac", "cardiovascular", "myocardial", "stroke", "hypertension"
        ],
        "Cancer/Oncology": [
            "cancer", "oncology", "tumor", "carcinoma", "leukemia", "lymphoma", "malignancy"
        ],
        "Diabetes/Endocrine": [
            "diabetes", "insulin", "glucose", "endocrine", "thyroid", "metabolic"
        ],
        "Mental Health": [
            "mental health", "depression", "anxiety", "psychiatric", "psychological", "suicide"
        ],
        "Infectious Disease": [
            "infection", "infectious", "sepsis", "antibiotic", "antimicrobial", "bacterial", "viral"
        ],
        "Pregnancy/Obstetrics": [
            "pregnancy", "pregnant", "maternal", "obstetric", "prenatal", "fetal"
        ],
        "Pediatrics": [
            "pediatric", "children", "infant", "adolescent", "neonatal"
        ],
        "Surgery/Procedures": [
            "surgery", "surgical", "operation", "procedure", "postoperative"
        ],
        "Drug Safety/Pharmacology": [
            "drug", "medication", "pharmaceutical", "adverse event", "pharmacology", "prescription"
        ],
        "Health Equity/Disparities": [
            "disparity", "disparities", "equity", "racial", "ethnic", "socioeconomic"
        ],
        "Machine Learning/AI": [
            "machine learning", "artificial intelligence", "deep learning", "neural network", "prediction model"
        ],
    }

    def assign_topic(row):
        text = f"{row['title']} {row['abstract']} {row['keywords']}".lower()

        matched_topics = []
        for topic, terms in topic_categories.items():
            if any(term in text for term in terms):
                matched_topics.append(topic)

        return "; ".join(matched_topics) if matched_topics else "Other/Unclassified"

    df["topic_category"] = df.apply(assign_topic, axis=1)
    return df


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════════

def create_visualizations(df):
    """Generate all analysis plots."""
    print("\n" + "=" * 70)
    print("GENERATING VISUALIZATIONS")
    print("=" * 70)

    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette("husl")

    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df_clean = df.dropna(subset=['year']).copy()
    df_clean['year'] = df_clean['year'].astype(int)

    plot_volume_and_if(df_clean, "trinetx", "TriNetX")
    plot_volume_and_if(df_clean, "cosmos", "Epic Cosmos")
    plot_combined_comparison(df_clean)
    plot_criticisms_by_year(df_clean)
    plot_geographic_distribution(df_clean)
    plot_topic_distribution(df_clean)

def plot_volume_and_if(df, source, title):
    df_source = df[df['source_query'].str.contains(source)].copy()
    if df_source.empty:
        print(f"  ⚠ No data for {title}, skipping")
        return

    stats = df_source.groupby('year').agg(
        volume=('pmid', 'count'),
        mean_if=('citedness_2yr', 'mean'),
    ).reset_index()

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.bar(stats['year'], stats['volume'], alpha=0.7, color='#0072B2', label='Article Count')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Article Count', color='#0072B2')
    ax1.tick_params(axis='y', labelcolor='#0072B2')

    ax2 = ax1.twinx()
    ax2.plot(stats['year'], stats['mean_if'], marker='o', color='#D55E00', label='Mean 2yr Citedness')
    ax2.set_ylabel('Mean 2-Year Citedness', color='#D55E00')
    ax2.tick_params(axis='y', labelcolor='#D55E00')

    plt.title(f'{title}: Annual Volume & Journal Citedness')
    fig.tight_layout()
    filename = f"{source}_volume_and_if.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    print(f"  ✓ Saved {filename}")
    plt.close()


def plot_combined_comparison(df):
    fig, ax = plt.subplots(figsize=(12, 6))

    for source, color, label in [
        ("trinetx", "#0072B2", "TriNetX"),
        ("cosmos",  "#D55E00", "Epic Cosmos"),
    ]:
        subset = df[df['source_query'].str.contains(source)]
        if subset.empty:
            continue
        counts = subset.groupby('year')['pmid'].count().reset_index()
        ax.plot(counts['year'], counts['pmid'], marker='o', color=color, label=label)

    ax.set_xlabel('Year')
    ax.set_ylabel('Article Count')
    ax.set_title('TriNetX vs Epic Cosmos: Annual Publication Volume')
    ax.legend()
    fig.tight_layout()
    plt.savefig("combined_comparison.png", dpi=150, bbox_inches='tight')
    print("  ✓ Saved combined_comparison.png")
    plt.close()


def plot_criticisms_by_year(df):
    flagged = df[df['pubmed_status'] != 'Clean'].copy()
    if flagged.empty:
        print("  ⚠ No flagged articles found, skipping")
        return

    pivot = (
        flagged.groupby(['year', 'pubmed_status'])['pmid']
        .count()
        .unstack(fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(kind='bar', ax=ax, alpha=0.8)
    ax.set_xlabel('Year')
    ax.set_ylabel('Article Count')
    ax.set_title('Flagged Articles by Year (Retractions, Errata, Comments)')
    ax.legend(title='Status')
    fig.tight_layout()
    plt.savefig("criticisms_by_year.png", dpi=150, bbox_inches='tight')
    print("  ✓ Saved criticisms_by_year.png")
    plt.close()


def plot_geographic_distribution(df):
    country_series = df['countries'].dropna().str.split('; ').explode()
    if country_series.empty:
        print("  ⚠ No country data found, skipping")
        return

    top_countries = country_series.value_counts().head(15)

    fig, ax = plt.subplots(figsize=(12, 6))
    top_countries.plot(kind='barh', ax=ax, color='#0072B2', alpha=0.7)
    ax.invert_yaxis()
    ax.set_xlabel('Article Count')
    ax.set_title('Top 15 Countries by Article Count')
    fig.tight_layout()
    plt.savefig("geographic_distribution.png", dpi=150, bbox_inches='tight')
    print("  ✓ Saved geographic_distribution.png")
    plt.close()


def plot_topic_distribution(df):
    topic_series = df['topic_category'].dropna().str.split('; ').explode()
    if topic_series.empty:
        print("  ⚠ No topic data found, skipping")
        return

    topic_counts = topic_series.value_counts()

    fig, ax = plt.subplots(figsize=(12, 6))
    topic_counts.plot(kind='barh', ax=ax, color='#D55E00', alpha=0.7)
    ax.invert_yaxis()
    ax.set_xlabel('Article Count')
    ax.set_title('Article Distribution by Topic Category')
    fig.tight_layout()
    plt.savefig("topic_distribution.png", dpi=150, bbox_inches='tight')
    print("  ✓ Saved topic_distribution.png")
    plt.close()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# REPORTING
# ═══════════════════════════════════════════════════════════════════════════

def generate_summary_report(df):
    print("\nSUMMARY REPORT")
    print(f"Total: {len(df)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════════════

def main():
    """Main execution function."""
    print("\n" + "═" * 70)
    print(" TriNetX & Epic Cosmos Literature Analysis")
    print("═" * 70)

    # Step 1: Fetch articles
    df = fetch_all_articles()
    df.to_parquet("01_raw_articles.parquet", index=False)

    # Step 2: Enrich with OpenAlex
    df = enrich_with_openalex(df)
    df.to_parquet("02_enriched_articles.parquet", index=False)

    # Step 3: Categorize topics
    df = categorize_topics(df)
    df.to_parquet("03_categorized_articles.parquet", index=False)

    # Step 4: Generate visualizations
    create_visualizations(df)

    # Step 5: Generate summary report
    generate_summary_report(df)

    # Step 6: Save final dataset
    df.to_csv("final_analysis.csv", index=False)
    df.to_parquet("final_analysis.parquet", index=False)

    print("\n" + "═" * 70)
    print(" Analysis Complete!")
    print("═" * 70)
    print("\nOutput files generated:")
    print("  • 01_raw_articles.parquet - Raw PubMed data")
    print("  • 02_enriched_articles.parquet - With OpenAlex metrics")
    print("  • 03_categorized_articles.parquet - With topic categories")
    print("  • final_analysis.csv - Final dataset (CSV)")
    print("  • final_analysis.parquet - Final dataset (Parquet)")
    print("  • trinetx_volume_and_if.png - TriNetX visualization")
    print("  • cosmos_volume_and_if.png - Cosmos visualization")
    print("  • combined_comparison.png - Comparison chart")
    print("  • criticisms_by_year.png - Criticism analysis")
    print("  • geographic_distribution.png - Country breakdown")
    print("  • topic_distribution.png - Topic breakdown")


if __name__ == "__main__":
    main()
